In [ ]:
!pip install transformers datasets -q
!pip install transformers -q
!pip install keras_nlp -q
!pip install datasets -q
!pip install huggingface-hub -q
!pip install nltk -q
!pip install rouge-score -q
!pip install huggingface_hub
!pip install rouge-score
!pip install datasets

import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer, DataCollatorForSeq2Seq
from transformers import AdamW
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from sklearn.metrics import average_precision_score
import gc
import string
from string import punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

model_name = "google/pegasus-xsum"
# model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def tokenize_inputs(inputs, tokenizer, max_length=512):
    return tokenizer(
        inputs,
        truncation=True,
        padding="longest",
        max_length=max_length,
        return_tensors="pt"
    )

class MemoryAugmentation:
    def __init__(self):
        self.n = 3
        self.max_memory_size = 1000
        self.memory = {}
        
    #N-Grams

    def remove_punctuation(self, text):
      if(type(text)==float):
        return text
      ans=""  
      for i in text:     
        if i not in string.punctuation:
          ans+=i    
      return ans

    def generate_N_grams(self, text):
        text = self.remove_punctuation(text)
        words=[word for word in text.split(" ") if word not in set(stopwords.words('english'))]  
        temp=zip(*[words[i:] for i in range(0,self.n)])
        ans=[' '.join(ngram) for ngram in temp]
        return ans

    def update_memory(self, text, input_data):
        ngrams = self.generate_N_grams(text)
        input_data = self.remove_punctuation(input_data)
        data = ""
        for word in input_data.split(" "):
            if word not in set(stopwords.words('english')):
                data += " "+word
        for ngram in ngrams:
            if ngram not in self.memory:
                self.memory[ngram] = set()
            self.memory[ngram].add(data)
            
            if len(self.memory[ngram]) > self.max_memory_size:
                list(self.memory[ngram]).pop(0)
        
    #TF-IDF

    def prepare_retrieval_data(self, corpus, query, top_k=2):
        if isinstance(corpus[0], list):
            corpus = [" ".join(doc) for doc in corpus]
            
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(corpus + [query])
        query_vector = tfidf_matrix[-1]
        
        similarities = cosine_similarity(query_vector, tfidf_matrix[:-1])
        top_k_indices = similarities.argsort()[0][-top_k:][::-1]
        top_k_docs = [corpus[idx] for idx in top_k_indices]
        
        return top_k_docs

    def get_relevant_data(self, text):
        ngrams = self.generate_N_grams(text)
        relevant_data = []
        output = []
        for ngram in ngrams:
            if ngram in self.memory:
                relevant_data.extend(self.memory[ngram])
                output = self.prepare_retrieval_data(list(set(relevant_data)), text)
                
        return output
    
external_memory = MemoryAugmentation()

In [ ]:
from transformers import AdamW
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import average_precision_score
from datasets import load_dataset

import pandas as pd
from datasets import Dataset

# CNN-DailyNews
data = load_dataset("cnn_dailymail", "3.0.0")

train_df = data["train"]

sub_dataset = train_df.train_test_split(train_size=0.001, seed=42)

split = sub_dataset["train"].train_test_split(train_size=0.6, seed=42)

train_dataset = split["train"]
val_dataset = split["test"]
test_dataset = val_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = test_dataset["train"]
test_dataset = test_dataset["test"]

print(f"\nTrain: {train_dataset}\n")
print(f"\nVal: {val_dataset}\n")
print(f"\nTest: {test_dataset}\n")


# Load CSV file
# csv_path = "/kaggle/input/news-dataset/New_Synthetic News Dataset and Model Details - Synthetic News Dataset.csv"
# data = pd.read_csv(csv_path)

# dataset = Dataset.from_pandas(data)

# split = dataset.train_test_split(train_size=0.8, seed=42)

# train_dataset = split["train"]
# val_dataset = split["test"]
# test_dataset = val_dataset.train_test_split(test_size=0.5, seed=42)
# val_dataset = test_dataset["train"]
# test_dataset = test_dataset["test"]

# print(f"\nTrain: {train_dataset}\n")
# print(f"\nVal: {val_dataset}\n")
# print(f"\nTest: {test_dataset}\n")

In [ ]:
from transformers import get_scheduler
import torch

import torch
from transformers import get_scheduler

def train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, epochs=1):
    model.to(device)
    model.train()
    
    num_training_steps = epochs * len(train_dataset)
    
    # Learning rate scheduler
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    for epoch in range(epochs):
        epoch_loss = 0
        for batch in train_dataset:
            # CNN-DailyMail News

            context = batch["article"]
            reference = batch["highlights"]

            retrieved_data = external_memory.get_relevant_data(context)
            
            # Custom CSV Data

            # category = batch["Category"]
            # query = batch["Headline"]
            # context = batch["Content"]
            # reference = batch["Human Summary"]
            
            # Prepare input text
            # input_text = f"{category}\n{query}\n{context}"
            input_text = context

            if len(retrieved_data) != 0:
                input_text = retrieved_data[0] + input_text

            data = context[:150]

            external_memory.update_memory(context, data)
            
            # Tokenize inputs and targets
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids.to(device)
            
            # Handle padding tokens
            labels[labels == tokenizer.pad_token_id] = -100
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            
            # Backpropagation
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Optimizer and scheduler steps
            optimizer.step()
            lr_scheduler.step()
            
            epoch_loss += loss.item()
            
        avg_epoch_loss = epoch_loss / len(train_dataset)
        print(f"Epoch {epoch+1} Training Loss = {avg_epoch_loss:.4f}")

        # Validate model after each epoch
        val_loss = evaluate_model(val_dataset, model, tokenizer, device)
        print(f"Epoch {epoch+1} Validation Loss = {val_loss:.4f}") 
    
    print("Training Completed")

def evaluate_model(val_dataset, model, tokenizer, device="cuda"):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in val_dataset:
            # CNN-DailyMail News

            context = batch["article"]
            reference = batch["highlights"]

            retrieved_data = external_memory.get_relevant_data(context)
            
            
            # Custom CSV Data
            
            # category = batch["Category"]
            # query = batch["Headline"]
            # context = batch["Content"]
            # reference = batch["Human Summary"]

            input_text = context

            if len(retrieved_data) != 0:
                input_text = retrieved_data[0] + input_text

            data = context[:150]

            external_memory.update_memory(context, data)

            # Tokenize inputs and targets
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids.to(device)

            # Handle padding tokens
            labels[labels == tokenizer.pad_token_id] = -100
            
            # Forward pass
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

    avg_val_loss = total_loss / len(val_dataset)
    return avg_val_loss

In [ ]:
import torch
from rouge_score import rouge_scorer

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics import precision
from collections import Counter

def calculate_metrics_on_csv(test_dataset, model, tokenizer, device="cuda"):
    model.eval()
    model.to(device)

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_dataset:  
            # CNN-DailyMail News

            context = batch["article"]
            reference = batch["highlights"]

            retrieved_data = external_memory.get_relevant_data(context)
            
            
            # Custom CSV Data
            
            # category = batch["Category"]
            # query = batch["Headline"]
            # context = batch["Content"]
            # reference = batch["Human Summary"]

            input_text = context

            if len(retrieved_data) != 0:
                input_text = retrieved_data[0] + input_text
            
            data = context[:150]

            external_memory.update_memory(context, data)
            
            # Tokenize inputs and generate summaries
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
            generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

            # Collect predictions and references
            predictions.append(generated_text)
            references.append(reference)

    rouge_scores = { "rouge1": [], "rouge2": [], "rougeL": [] }
    precision_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge_scores["rouge1"].append(scores["rouge1"].fmeasure)
        rouge_scores["rouge2"].append(scores["rouge2"].fmeasure)
        rouge_scores["rougeL"].append(scores["rougeL"].fmeasure)

        # **Average Precision Calculation (Word-Level)**
        ref_words = Counter(ref.split())
        pred_words = Counter(pred.split())
        common_words = sum((ref_words & pred_words).values())  # Correct words retrieved
        total_predicted_words = sum(pred_words.values())  # Total words in generated summary
        precision_score = common_words / total_predicted_words if total_predicted_words > 0 else 0
        precision_scores.append(precision_score)

        ref_tokens = [ref.split()]  # BLEU expects a list of reference token lists
        pred_tokens = pred.split()

    avg_rouge_scores = {key: sum(values) / len(values) for key, values in rouge_scores.items()}
    avg_precision = sum(precision_scores) / len(precision_scores)

    print(f"Average ROUGE Scores: {avg_rouge_scores}")
    print(f"Average Precision: {avg_precision:.4f}")

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
train_loss = train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, epochs=3)

In [ ]:
calculate_metrics_on_csv(test_dataset, model, tokenizer)

In [ ]:
def Generate_Result(input_text, model, tokenizer):
    retrieved_data = external_memory.get_relevant_data(input_text)
    print(f"Retrieved Data: {retrieved_data}")

    data = input_text[:150]
    external_memory.update_memory(input_text, data)

    if len(retrieved_data) != 0:
        input_text = retrieved_data[0] + input_text
    
    inputs = tokenize_inputs(input_text, tokenizer)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(inputs["input_ids"], max_length=128)

    return outputs

In [ ]:
text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

outputs = Generate_Result(text, model, tokenizer)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

In [ ]:
text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

outputs = Generate_Result(text, model, tokenizer)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

In [ ]:
text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

outputs = Generate_Result(text, model, tokenizer)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

In [ ]:
text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

outputs = Generate_Result(text, model, tokenizer)
print(f"Output: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")